In [48]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

In [19]:
data = load_breast_cancer()
X, y = data['data'], data['target']
X.shape, y.shape

((569, 30), (569,))

In [20]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [21]:
# Standardize inputs and target
x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train)
X_test = x_scaler.transform(X_test)

In [22]:
# Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.int64)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.int64)

In [39]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 5)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(5, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x

In [40]:
model = MLP(input_dim=30)

In [41]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [43]:
# Training loop
epochs = 50
model.train()

for epoch in range(epochs):

    y_hat = model(X_train)
    loss = criterion(y_hat, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch + 1}/{epochs} | Train CE: {loss.item():.2f}")

Epoch 1/50 | Train CE: 0.54
Epoch 2/50 | Train CE: 0.53
Epoch 3/50 | Train CE: 0.53
Epoch 4/50 | Train CE: 0.52
Epoch 5/50 | Train CE: 0.52
Epoch 6/50 | Train CE: 0.51
Epoch 7/50 | Train CE: 0.51
Epoch 8/50 | Train CE: 0.50
Epoch 9/50 | Train CE: 0.50
Epoch 10/50 | Train CE: 0.49
Epoch 11/50 | Train CE: 0.49
Epoch 12/50 | Train CE: 0.48
Epoch 13/50 | Train CE: 0.48
Epoch 14/50 | Train CE: 0.48
Epoch 15/50 | Train CE: 0.47
Epoch 16/50 | Train CE: 0.47
Epoch 17/50 | Train CE: 0.46
Epoch 18/50 | Train CE: 0.46
Epoch 19/50 | Train CE: 0.45
Epoch 20/50 | Train CE: 0.45
Epoch 21/50 | Train CE: 0.44
Epoch 22/50 | Train CE: 0.44
Epoch 23/50 | Train CE: 0.43
Epoch 24/50 | Train CE: 0.43
Epoch 25/50 | Train CE: 0.43
Epoch 26/50 | Train CE: 0.42
Epoch 27/50 | Train CE: 0.42
Epoch 28/50 | Train CE: 0.41
Epoch 29/50 | Train CE: 0.41
Epoch 30/50 | Train CE: 0.40
Epoch 31/50 | Train CE: 0.40
Epoch 32/50 | Train CE: 0.40
Epoch 33/50 | Train CE: 0.39
Epoch 34/50 | Train CE: 0.39
Epoch 35/50 | Train CE:

In [44]:
# Evaluation
model.eval()

with torch.no_grad():
    y_hat_test = model(X_test)
    y_pred = torch.argmax(y_hat_test, dim=1)

accuracy = (y_pred == y_test).float().mean()

print(f"Test accuracy: {accuracy.item():.3f}")

Test accuracy: 0.965


In [49]:
def classification_metrics(target, pred):
    tn, fp, fn, tp = metrics.confusion_matrix(target, pred).ravel()
    acc = (tp + tn) / (tn + fp + fn + tp)
    sen = tp / (tp + fn)
    spc = tn / (tn + fp)
    prc = tp / (tp + fp)
    return acc, sen, spc, prc

In [50]:
classification_metrics(y_test, y_pred)

(0.9649122807017544,
 0.9577464788732394,
 0.9767441860465116,
 0.9855072463768116)